# [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/pdf/1412.6550)

## Introduction

Turning pre-trained models into task-specific models via fine-tuning is expensive, since we train and store all the weights for each task.

There was two main methods of addressing the above issue prior to LoRA:

1] **[Adapters](https://arxiv.org/pdf/1902.00751)**: Insert small bottleneck modules within each layer of a frozen pretrained model, which project hidden states into a lower-dimensional space and then project them back to the original hidden size. This enables task-specific adaptation with a minimal number of additional parameters.

2a] **[Prefix-tuning](https://arxiv.org/abs/2101.00190)**: Learn continuous task-specific vectors that are injected as additional key/value pairs in the attention mechanism at each layer, while keeping the pretrained model frozen. This enables input sequence to attend to learned task-specific memory.

2b] **[Prompt-tuning](https://arxiv.org/pdf/2104.08691)**: Learn task-specific continuous embeddings that are prepended to the input sequence as "soft tokens", with all pre-trained weights frozen.

**Gap:** Adapters introduce inference latency by extending model depth, while prompt-tuning reduces the model's usable sequence length.

**Improvement:** LoRA enables tailoring pre-trained models for specific tasks without additional inference latency and without eating into the mode's usable sequence length.

## Approach

LoRA training learns the change in weights ($\Delta{W}$) needed for a pre-trained model to adapt to a dataset without actually training the model's weights.

But directly learning $\Delta{W}$ requires learning same number of parameters as learning $W$. To train for $\Delta{W}$ more efficiently, they approximate $\Delta{W}$ using a low-rank factorization:

$$
\Delta{W} = AB
$$

Let $\Delta{W}$ be dimensions $n \ * \ m$. Then, $A$ and $B$ are dimensions $n \ * \ k$ and $k \ * \ m$, respectively. This constrains $\Delta{W}$ to have rank at most $k$.

If $\#params (\Delta{W}) = 100 * 50 = 5,000$, then a rank of at most $k = 3$ would reduce the training parameters from $5000$ to $\#params (A) + \#params (B) = 100 * 3 + 3 * 50 = 450$.

So for pre-trained weights, $W_0$, LoRA training updates standard training like so: 

$$h = W_0x \rightarrow h = (W_0 + \Delta{W})x = (W_0 + AB)x = W_0x + ABx$$

## Result

- Compared to GPT-3 175B full fine-tuning, LoRA can reduce trainable parameters by 10,000 times and the GPU memory requirement by 3 times.
- Adapter-based methods contain 20% more latency than LoRA on single instance inference. LoRA adds no latency compared to fine-tuning since $AB$ can be absorbed into the weights for inference.
- Showed that LoRA performs on-par or better than fine-tuning and adapters in model quality.

## Application